In [ ]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [ ]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [ ]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [ ]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [ ]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [ ]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [ ]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [ ]:
confusion_matrix_and_metrics(y_train, y_pred_train)

### Check results on the test set (new data not yet seen by the model)

In [ ]:
confusion_matrix_and_metrics(y_test, y_pred_test)

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

My confidence in this model's ability to predict a bot is **low**, and the misclassification rate alone hides why.

On the test set the model misclassifies 10.9% of accounts, which sounds reasonable until you compare it to the right baseline. Only 12.6% of the test accounts are actually bots, so a "model" that ignored every input and labeled *every account human* would score 12.6% misclassification. My model gets 10.9%. All that tuning bought about 1.7 percentage points over a rule that does no work at all.

The reason is class imbalance: the model earns almost its entire score by correctly identifying humans, which is the easy and common case. On the thing we actually built it to do, it performs poorly. It catches **30 of 126 bots (23.8% recall)** and misses the other 96, a false negative rate of 76.2%. Three out of every four bots pass through undetected.

This is also why the original hyperparameters (`max_depth=8`) were deceptive. That model scored 13.2% misclassification, meaning it was *worse than guessing "human" every single time*, and its near-zero training error (0.03%) made it look excellent on the data it had already seen. Reducing depth to 2 closed the train/test gap from 0.132 to 0.005 and genuinely improved generalization, but no amount of hyperparameter tuning fixed the underlying problem: five behavioral features (dominated by `account_age_days` at 0.51 and `tweets_per_day` at 0.38) do not cleanly separate bots from humans, because plenty of real people are new to a platform and post frequently.

So I would not deploy this to take automated action on accounts. A single aggregate accuracy number is the wrong measure for an imbalanced problem, and reporting "89% accurate" here would be technically true and substantively misleading.

### What are potential ramifications of false positives from the model?

A false positive is a **real person wrongly labeled a bot**, and the cost falls on someone who did nothing wrong.

Depending on how the platform wires up the output, that person may be suspended, shadow-banned, locked out pending ID verification, or have their reach quietly throttled. They typically lose access to their own account, contacts, and archive, often with no clear explanation and an appeals process that is slow, automated, and hard to escalate. The burden of proof lands on the accused user, who has to prove a negative.

My model's false positive rate looks small at 1.5% (13 accounts out of 874 humans in the test set), but that framing is misleading at platform scale. Applied to a service with 100 million human accounts, a 1.5% false positive rate is roughly **1.5 million people incorrectly flagged**. "Rare" errors are still enormous in absolute terms once you multiply by the user base.

The errors also will not be randomly distributed. The model leans on account age and posting frequency, so the users most likely to be flagged are new accounts that post a lot: people joining during a breaking news event, activists and journalists covering a crisis, fan communities, small businesses starting out, and users whose posting rhythms differ from the training population. These are disproportionately newcomers and people organizing around time-sensitive events, which is exactly the speech a platform can least afford to silence by accident. There is a real chilling effect, since users who see peers wrongly banned learn to self-censor normal enthusiastic behavior.

Because the harm is concentrated on individuals who cannot easily contest it, I would want a human review step before any irreversible enforcement, rather than letting this model act on its own.

### What are potential ramifications of false negatives from the model?

A false negative is a **bot the model fails to catch**, and this is where my model fails most often: 96 of 126 bots in the test set slipped through, a false negative rate of 76.2%.

The harms here are diffuse rather than individual. Undetected bots continue to spread spam, scams, and phishing links; amplify disinformation and coordinated political messaging; manipulate trending topics and recommendation algorithms so that manufactured consensus looks organic; inflate engagement metrics that advertisers pay for and that researchers treat as real human behavior; and drown out genuine conversation. Each missed bot is small on its own, but bot activity is coordinated by design, so the aggregate effect on public discourse is substantial.

There is a second, subtler harm: a deployed detector creates a **false sense of security**. If the platform advertises that it runs bot detection, users reasonably extend more trust to accounts that were not flagged. A model that misses three quarters of bots effectively launders those bots as legitimate. In that sense, a weak detector presented as a strong one can be worse than no detector at all, because it changes user behavior without earning the trust it receives.

Comparing the two error types, my model is heavily skewed toward false negatives (96) over false positives (13). It is a conservative model that rarely accuses an innocent user but lets most bots operate freely. That balance is a **choice, not a fact about the data**, and it is adjustable: lowering the decision threshold from 0.5 to 0.3 nearly doubles recall from 23.8% to 45.2%, at the cost of raising the false positive rate from 1.5% to 6.3% and overall misclassification from 0.109 to 0.124.

That tradeoff is the real lesson. Optimizing for the lowest misclassification rate, which is what the assignment asks, pushes the model toward catching almost no bots, because ignoring the rare class is the cheapest way to be right most often. Minimizing aggregate error and actually detecting bots are competing objectives here, and deciding where to sit between them is an ethical and editorial judgment about who absorbs the harm, wrongly punished users or the broader information ecosystem. It should not be settled silently by whichever setting a validation script happens to score best.